# Projet — Prédiction du risque d'accident de vélo

**Analyse exploratoire des données (EDA)**

Ce notebook constitue la première étape du projet : comprendre, nettoyer et explorer le jeu de données BAAC (Bulletin d'Analyse des Accidents Corporels) restreint aux accidents impliquant un vélo. L'objectif final est de prédire la gravité d'un accident (grave vs non-grave) à partir des conditions de l'accident.

## Plan du notebook

1. **Imports et chargement** — bibliothèques, lecture du fichier
2. **Dictionnaire des variables** — référence des codes BAAC
3. **Filtrage temporel** — restriction à 2018-2024 pour homogénéité méthodologique
4. **Qualité des données** — valeurs manquantes, sentinelles, doublons, cohérence
5. **Nettoyage et création de variables** — pipeline de préparation
6. **Analyse univariée** — distributions des variables principales
7. **Analyse de la cible `grav`** — construction de la cible binaire
8. **Analyse bivariée** — variables explicatives × cible
9. **Tests statistiques** — chi², Mann-Whitney, Cramér's V
10. **Analyse temporelle approfondie** — saisonnalité, heure × jour
11. **Analyse géographique** — départements, urbain/rural, carte
12. **Corrélations** — Pearson, Spearman, Cramér's V
13. **Synthèse, biais et limites**

## Choix méthodologique central

L'ONISR signale plusieurs ruptures dans la nomenclature BAAC :
- en 2018-2019, la codification a été révisée (apparition du code `-1` pour "non renseigné")
- l'indicateur "blessé hospitalisé" (`grav=3`) n'est plus comparable avant/après 2018 en raison d'un changement du processus de saisie par les forces de l'ordre

**Décision retenue : restreindre l'analyse à la période 2018-2024.** Cela réduit la taille de l'échantillon mais garantit l'homogénéité des indicateurs, ce qui est indispensable pour entraîner un modèle dont les coefficients restent interprétables. Les analyses sur la plage complète (2005-2024) sont conservées uniquement à titre descriptif pour mettre en évidence ces ruptures.

**Source de la nomenclature BAAC :** ONISR, *Description des bases de données annuelles des accidents corporels de la circulation routière (2005-2023)*, 23 octobre 2024.


## 1. Imports et configuration

In [ ]:
# Manipulation et calcul
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Statistiques
from scipy.stats import chi2_contingency, mannwhitneyu, spearmanr, pearsonr

# Affichage
from IPython.display import display, Markdown
import warnings
warnings.filterwarnings('ignore')

# Configuration globale
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Bibliothèques chargées.")

## 2. Chargement des données

Le fichier `accidentsVelo.csv` est une fusion des 4 rubriques BAAC (Caractéristiques, Lieux, Véhicules, Usagers), restreinte aux accidents impliquant au moins un vélo. Il a été produit à partir de la base ouverte data.gouv.fr et redistribué via Koumoul.

In [ ]:
df = pd.read_csv("/content/accidentsVelo.csv", low_memory=False)

print(f"Dimensions brutes : {df.shape[0]} lignes × {df.shape[1]} colonnes")
print(f"\nPremières lignes :")
display(df.head(3))
print(f"\nTypes de colonnes :")
print(df.dtypes.value_counts())

**Lecture.** Le jeu compte environ 80 000 lignes pour 39 colonnes. Une ligne représente un *usager-véhicule* impliqué dans un accident — un même accident peut donc apparaître plusieurs fois (un cycliste + une voiture = 2 lignes). Cette structure conditionne toutes les analyses : pour compter des accidents, il faudra dédoublonner sur `Num_Acc` ; pour analyser des victimes ou la gravité individuelle, on garde toutes les lignes.

## 3. Dictionnaire des variables

Les codes BAAC ne sont pas auto-explicatifs (ex. `lum=3` signifie "nuit sans éclairage"). Cette section définit les mappings utilisés dans l'ensemble du notebook pour rendre les visualisations lisibles.

Source officielle : [ONISR — Description des bases de données annuelles](https://www.onisr.securite-routiere.gouv.fr/sites/default/files/2024-10/Description%20des%20bases%20de%20donn%C3%A9es%20annuelles.pdf).

In [ ]:
# Caractéristiques de l'accident
mapping_grav = {1: 'Indemne', 2: 'Tué', 3: 'Blessé hospitalisé', 4: 'Blessé léger'}
mapping_lum = {1: 'Plein jour', 2: 'Crépuscule/aube', 3: 'Nuit sans éclairage',
               4: 'Nuit éclairage non allumé', 5: 'Nuit éclairage allumé'}
mapping_atm = {-1: 'Non renseigné', 1: 'Normale', 2: 'Pluie légère', 3: 'Pluie forte',
               4: 'Neige/grêle', 5: 'Brouillard', 6: 'Vent fort', 7: 'Temps éblouissant',
               8: 'Temps couvert', 9: 'Autre'}
mapping_agg = {1: 'Hors agglomération', 2: 'En agglomération'}
mapping_int = {1: 'Hors intersection', 2: 'Intersection en X', 3: 'Intersection en T',
               4: 'Intersection en Y', 5: '+ de 4 branches', 6: 'Giratoire',
               7: 'Place', 8: 'Passage à niveau', 9: 'Autre intersection'}
mapping_col = {-1: 'Non renseigné', 1: '2 véh. - frontale', 2: '2 véh. - par l\'arrière',
               3: '2 véh. - par le côté', 4: '3+ véh. - en chaîne',
               5: '3+ véh. - collisions multiples', 6: 'Autre collision', 7: 'Sans collision'}

# Lieux
mapping_catr = {1: 'Autoroute', 2: 'Route nationale', 3: 'Route départementale',
                4: 'Voie communale', 5: 'Hors réseau public',
                6: 'Parking ouvert', 7: 'Routes métropole urbaine', 9: 'Autre'}
mapping_circ = {-1: 'Non renseigné', 1: 'Sens unique', 2: 'Bidirectionnelle',
                3: 'Chaussées séparées', 4: 'Voies d\'affectation variable'}
mapping_prof = {-1: 'Non renseigné', 1: 'Plat', 2: 'Pente', 3: 'Sommet de côte', 4: 'Bas de côte'}
mapping_plan = {-1: 'Non renseigné', 1: 'Rectiligne', 2: 'Courbe à gauche',
                3: 'Courbe à droite', 4: 'En S'}
mapping_surf = {-1: 'Non renseigné', 1: 'Normale', 2: 'Mouillée', 3: 'Flaques',
                4: 'Inondée', 5: 'Enneigée', 6: 'Boue', 7: 'Verglacée',
                8: 'Corps gras', 9: 'Autre'}
mapping_infra = {-1: 'Non renseigné', 0: 'Aucun', 1: 'Souterrain/tunnel',
                 2: 'Pont/autopont', 3: 'Bretelle', 4: 'Voie ferrée',
                 5: 'Carrefour aménagé', 6: 'Zone piétonne', 7: 'Zone de péage',
                 8: 'Chantier', 9: 'Autres'}
mapping_situ = {-1: 'Non renseigné', 0: 'Aucun', 1: 'Sur chaussée',
                2: 'Sur BAU', 3: 'Sur accotement', 4: 'Sur trottoir',
                5: 'Sur piste cyclable', 6: 'Sur autre voie spéciale', 8: 'Autres'}

# Véhicules
mapping_obs = {-1: 'Non renseigné', 0: 'Sans objet', 1: 'Véh. en stationnement',
               2: 'Arbre', 3: 'Glissière métal', 4: 'Glissière béton', 5: 'Autre glissière',
               6: 'Bâtiment/mur', 7: 'Signalisation verticale', 8: 'Poteau',
               9: 'Mobilier urbain', 10: 'Parapet', 11: 'Îlot/refuge',
               12: 'Bordure trottoir', 13: 'Fossé/talus', 14: 'Autre obs. chaussée',
               15: 'Autre obs. trottoir', 16: 'Sortie sans obstacle', 17: 'Buse'}
mapping_obsm = {-1: 'Non renseigné', 0: 'Aucun', 1: 'Piéton', 2: 'Véhicule',
                4: 'Véhicule sur rail', 5: 'Animal domestique', 6: 'Animal sauvage', 9: 'Autre'}
mapping_choc = {-1: 'Non renseigné', 0: 'Aucun', 1: 'Avant', 2: 'Avant droit',
                3: 'Avant gauche', 4: 'Arrière', 5: 'Arrière droit', 6: 'Arrière gauche',
                7: 'Côté droit', 8: 'Côté gauche', 9: 'Chocs multiples'}

# Usagers
mapping_sexe = {1: 'Masculin', 2: 'Féminin'}
mapping_trajet = {-1: 'Non renseigné', 0: 'Non renseigné', 1: 'Domicile-travail',
                  2: 'Domicile-école', 3: 'Courses', 4: 'Pro', 5: 'Promenade/loisirs', 9: 'Autre'}

# Dictionnaire global pour usage programmatique
mappings = {
    'grav': mapping_grav, 'lum': mapping_lum, 'atm': mapping_atm, 'agg': mapping_agg,
    'int': mapping_int, 'col': mapping_col, 'catr': mapping_catr, 'circ': mapping_circ,
    'prof': mapping_prof, 'plan': mapping_plan, 'surf': mapping_surf,
    'infra': mapping_infra, 'situ': mapping_situ, 'obs': mapping_obs,
    'obsm': mapping_obsm, 'choc': mapping_choc, 'sexe': mapping_sexe, 'trajet': mapping_trajet
}

print(f"{len(mappings)} variables mappées.")

**Vérification automatique des modalités effectivement présentes.** Avant d'utiliser ces mappings, on contrôle qu'aucun code de la base n'est hors nomenclature documentée — ce qui révèlerait des évolutions BAAC non prises en compte.

In [ ]:
for var, mapping in mappings.items():
    if var not in df.columns:
        continue
    valeurs_data = set(df[var].dropna().unique())
    valeurs_doc = set(mapping.keys())
    non_documentees = valeurs_data - valeurs_doc
    if non_documentees:
        print(f"⚠️  {var} : codes hors nomenclature → {non_documentees}")
    else:
        print(f"✓  {var}")

Les codes hors nomenclature éventuels seront traités comme manquants au moment du nettoyage. Si un code apparaît avec une fréquence non négligeable, on le documente comme une zone d'incertitude.

## 4. Filtrage temporel : restriction à 2018-2024

### 4.1 Visualisation de la rupture méthodologique

Avant de filtrer, on met en évidence pourquoi on filtre. On affiche le taux de "blessé hospitalisé" (`grav=3`) par année sur l'ensemble des données : un changement brutal en 2018-2019 confirmera la rupture documentée par l'ONISR.

In [ ]:
# Sur la base brute (avant filtrage) : taux des 4 modalités de grav par année
grav_par_an_full = df.groupby('an')['grav'].value_counts(normalize=True).unstack() * 100

fig, ax = plt.subplots(figsize=(12, 5))
for g in [1, 2, 3, 4]:
    if g in grav_par_an_full.columns:
        ax.plot(grav_par_an_full.index, grav_par_an_full[g],
                marker='o', label=f"{g} - {mapping_grav[g]}")
ax.axvline(2018.5, ls='--', color='red', alpha=0.6, label='Rupture BAAC 2018-2019')
ax.set_xlabel('Année')
ax.set_ylabel('% des usagers')
ax.set_title('Évolution annuelle de la répartition de la gravité (base brute)')
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

**Lecture.** On observe clairement une discontinuité autour de 2018-2019 sur les modalités `2 - Tué` et `3 - Blessé hospitalisé` : le taux de blessés hospitalisés baisse fortement après 2018, non pas parce que les cyclistes sont mieux protégés, mais parce que les forces de l'ordre ont changé leur façon de classer les blessés. C'est exactement la rupture que l'ONISR signale dans sa documentation officielle. **Inclure les données 2005-2017 introduirait un biais qui ferait conclure faussement que "la gravité diminue avec le temps", alors que c'est un artéfact de saisie.**

### 4.2 Application du filtre

In [ ]:
df_full = df.copy()  # On garde la base complète sous la main si besoin

df = df[df['an'] >= 2018].reset_index(drop=True)
print(f"Lignes avant filtrage : {len(df_full):,}")
print(f"Lignes après filtrage 2018+ : {len(df):,}")
print(f"Perte : {(1 - len(df)/len(df_full))*100:.1f}%")
print(f"\nRépartition par année après filtrage :")
print(df['an'].value_counts().sort_index())

**Conséquence.** On perd une part importante des observations, mais ce qui reste est homogène en termes de protocole de saisie. Tous les modèles entraînés par la suite seront calibrés sur cette plage. Pour la production future, le modèle ne devra être appliqué qu'à des données saisies selon le BAAC 2017+ (ce qui est désormais le cas par défaut).

## 5. Qualité des données

### 5.1 Aperçu général

In [ ]:
print(f"Dimensions : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
print(f"Mémoire : {df.memory_usage(deep=True).sum() / 1024**2:.1f} Mo")
print(f"\nNombre d'accidents uniques (Num_Acc) : {df['Num_Acc'].nunique():,}")
print(f"Moyenne lignes / accident : {len(df) / df['Num_Acc'].nunique():.2f}")

### 5.2 Valeurs manquantes (NaN classiques)

In [ ]:
missing = pd.DataFrame({
    'n_missing': df.isna().sum(),
    'pct_missing': (df.isna().sum() / len(df) * 100).round(2)
}).sort_values('pct_missing', ascending=False)

print("Colonnes avec au moins un NaN :")
display(missing[missing['pct_missing'] > 0])

# Visualisation
to_plot = missing[missing['pct_missing'] > 0].sort_values('pct_missing')
if len(to_plot) > 0:
    fig, ax = plt.subplots(figsize=(10, max(3, len(to_plot)*0.3)))
    ax.barh(to_plot.index, to_plot['pct_missing'], color='steelblue')
    ax.set_xlabel('% de valeurs manquantes')
    ax.set_title('Taux de NaN par colonne (après filtrage 2018+)')
    plt.tight_layout()
    plt.show()
else:
    print("Aucun NaN classique détecté.")

**Lecture.** Le BAAC est une saisie obligatoire par les forces de l'ordre, donc peu de NaN classiques. La majorité des "manquants" sont en réalité encodés via des sentinelles (`-1`, `0`, cellule vide convertie en string), à traiter ci-dessous.

### 5.3 Valeurs sentinelles (`-1` = "non renseigné")

In [ ]:
vars_minus1 = ['atm', 'col', 'circ', 'prof', 'plan', 'surf', 'infra',
               'situ', 'obs', 'obsm', 'choc']

print("Taux de '-1' (non renseigné) par variable :")
sentinelles = []
for v in vars_minus1:
    if v in df.columns:
        n = (df[v] == -1).sum()
        pct = n / len(df) * 100
        sentinelles.append({'variable': v, 'n_minus1': n, 'pct': round(pct, 2)})
        print(f"  {v:8s} : {n:>6} ({pct:5.2f}%)")

pd.DataFrame(sentinelles).set_index('variable')

In [ ]:
# Géolocalisation : 0 = manquant (pas l'équateur)
df['lat'] = pd.to_numeric(df['lat'].astype(str).str.replace(',', '.'), errors='coerce')
df['long'] = pd.to_numeric(df['long'].astype(str).str.replace(',', '.'), errors='coerce')

n_geo_missing = ((df['lat'] == 0) | (df['long'] == 0) |
                 df['lat'].isna() | df['long'].isna()).sum()
print(f"Géolocalisation manquante (lat ou long = 0/NaN) : "
      f"{n_geo_missing:,} ({n_geo_missing/len(df)*100:.1f}%)")

**Lecture.** Les variables `obs` (obstacle fixe heurté) et `obsm` (obstacle mobile) ont un taux de `-1` souvent élevé, parce qu'elles sont conditionnelles : si le cycliste n'a heurté aucun obstacle, le champ peut rester vide. Ce n'est donc pas systématiquement un défaut de saisie. Pour les autres variables (`atm`, `surf`, `lum`...), un taux > 5 % de `-1` mérite une mention dans la section "limites".

La géolocalisation manquante (~10 % attendus) est un biais classique du BAAC, particulièrement marqué en zone rurale.

### 5.4 Doublons

In [ ]:
# Doublons exacts (toutes colonnes identiques)
n_dup_full = df.duplicated().sum()
print(f"Doublons exacts (lignes complètement identiques) : {n_dup_full}")

# Structure : un Num_Acc peut apparaître plusieurs fois (plusieurs usagers/véhicules)
print(f"\nNombre de lignes : {len(df):,}")
print(f"Nombre d'accidents uniques : {df['Num_Acc'].nunique():,}")

# Distribution du nombre de lignes par accident
dist_lignes = df.groupby('Num_Acc').size().value_counts().sort_index()
print(f"\nDistribution du nombre de lignes par accident :")
print(dist_lignes.head(10))

**Lecture.** Un même `Num_Acc` peut apparaître plusieurs fois : ce n'est **pas** un bug. Chaque ligne représente un couple usager-véhicule. Un accident vélo-voiture avec 2 occupants dans la voiture donne 3 lignes. Les doublons exacts (toutes colonnes identiques) sont en revanche suspects et seraient à supprimer.

**Implication méthodologique.** Pour les analyses au niveau "accident" (top départements, conditions météo), il faudra dédoublonner : `df.drop_duplicates('Num_Acc')`. Pour les analyses au niveau "usager" (gravité individuelle, équipement), on garde toutes les lignes. Ce choix est explicité à chaque section concernée.

### 5.5 Cohérence des valeurs

In [ ]:
# Âge
print("=== Âge ===")
print(df['age'].describe().round(1))
print(f"\nÂge < 0    : {(df['age'] < 0).sum()}")
print(f"Âge > 100   : {(df['age'] > 100).sum()}")
print(f"Âge = 0    : {(df['age'] == 0).sum()}  (probable manquant codé en 0)")

In [ ]:
# Heure
print("\n=== Heure (hrmn) ===")
print("Échantillon :", df['hrmn'].dropna().sample(5, random_state=1).values)
df['heure'] = pd.to_numeric(df['hrmn'].astype(str).str.split(':').str[0], errors='coerce')
print(f"Heures hors plage [0,23] : {((df['heure'] < 0) | (df['heure'] > 23)).sum()}")

In [ ]:
# Nombre de voies (nbv stocké en string avec '00', '01'...)
df['nbv'] = pd.to_numeric(df['nbv'], errors='coerce')
print("\n=== Nombre de voies ===")
print(df['nbv'].describe().round(1))
print(f"nbv = 0  : {(df['nbv'] == 0).sum()}  (probable manquant)")
print(f"nbv > 10 : {(df['nbv'] > 10).sum()}  (suspect)")

**Lecture.**
- **Âge** : la valeur 0 est ambiguë — soit un nourrisson cycliste (improbable), soit un manquant codé en 0. On retient cette interprétation. Les âges > 100 sont rares mais possibles ; les âges négatifs sont des erreurs de saisie.
- **Heure** : le format `HH:MM` est respecté. La conversion en `heure` (entier 0-23) est utilisée pour les analyses temporelles.
- **Nombre de voies** : `nbv = 0` n'a pas de sens physique, c'est probablement un manquant. Les valeurs très élevées (> 10) sont aberrantes et seront traitées dans le nettoyage.

## 6. Nettoyage et création de variables

On consolide ici les opérations de préparation pour disposer d'un DataFrame `df` propre, prêt pour l'analyse.

In [ ]:
# 6.1 Conversion des sentinelles -1 en NaN pour faciliter les analyses
vars_a_nettoyer = ['atm', 'col', 'circ', 'prof', 'plan', 'surf', 'infra',
                   'situ', 'obs', 'obsm', 'choc']
for v in vars_a_nettoyer:
    if v in df.columns:
        df.loc[df[v] == -1, v] = np.nan

# Pour trajet, -1 ET 0 codent "non renseigné"
if 'trajet' in df.columns:
    df.loc[df['trajet'].isin([-1, 0]), 'trajet'] = np.nan

# 6.2 Âge : on garde tel quel mais on note les âges = 0 comme suspects
df['age_clean'] = df['age'].where((df['age'] > 0) & (df['age'] <= 100), np.nan)

# 6.3 nbv = 0 ou > 10 → NaN
df['nbv_clean'] = df['nbv'].where(df['nbv'].between(1, 10, inclusive='both'), np.nan)

# 6.4 Coordonnées GPS : 0 → NaN (déjà fait pour la conversion, on s'assure)
df.loc[df['lat'] == 0, 'lat'] = np.nan
df.loc[df['long'] == 0, 'long'] = np.nan

# 6.5 Création de la variable date complète
df['date_dt'] = pd.to_datetime(df['date'], errors='coerce')

# 6.6 Création de variables temporelles dérivées
df['heure'] = pd.to_numeric(df['hrmn'].astype(str).str.split(':').str[0], errors='coerce')

# Plage horaire
def categorise_heure(h):
    if pd.isna(h): return np.nan
    if 6 <= h < 9: return 'matin (6-9h)'
    if 9 <= h < 12: return 'milieu_matinée (9-12h)'
    if 12 <= h < 14: return 'midi (12-14h)'
    if 14 <= h < 17: return 'après-midi (14-17h)'
    if 17 <= h < 20: return 'soir (17-20h)'
    if 20 <= h < 23: return 'nuit_début (20-23h)'
    return 'nuit_profonde (23-6h)'

df['plage_horaire'] = df['heure'].apply(categorise_heure)

# Saison
def categorise_mois(m):
    if m in ['décembre', 'janvier', 'février']: return 'hiver'
    if m in ['mars', 'avril', 'mai']: return 'printemps'
    if m in ['juin', 'juillet', 'août']: return 'été'
    return 'automne'
df['saison'] = df['mois'].apply(categorise_mois)

# Week-end
df['weekend'] = df['jour'].isin(['samedi', 'dimanche']).astype(int)

# 6.7 Tranches d'âge
df['age_tranche'] = pd.cut(df['age_clean'],
                            bins=[0, 15, 25, 35, 50, 65, 80, 120],
                            labels=['<15', '15-25', '25-35', '35-50', '50-65', '65-80', '>80'])

print("Nettoyage terminé. Nouvelles colonnes créées :")
print(['lat', 'long', 'heure', 'age_clean', 'nbv_clean', 'date_dt',
       'plage_horaire', 'saison', 'weekend', 'age_tranche'])

**Note sur les choix de nettoyage.**
- On *ne supprime pas* les lignes avec des sentinelles : on les convertit en NaN, ce qui permet de les exclure variable par variable plutôt qu'en bloc. Une suppression totale ferait perdre des dizaines de % de l'échantillon.
- Les variables nettoyées sont suffixées `_clean` pour conserver les originales en cas de besoin de comparaison.
- Les variables temporelles dérivées (`plage_horaire`, `saison`, `weekend`) anticipent le feature engineering pour la modélisation.

## 7. Analyse univariée

### 7.1 Variables numériques

#### Âge — histogramme + boxplot + violon

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Histogramme
df['age_clean'].dropna().plot(kind='hist', bins=50, ax=axes[0],
                               color='steelblue', edgecolor='white')
axes[0].set_title('Distribution de l\'âge — histogramme')
axes[0].set_xlabel('Âge')

# Boxplot
axes[1].boxplot(df['age_clean'].dropna(), vert=True)
axes[1].set_title('Distribution de l\'âge — boxplot')
axes[1].set_ylabel('Âge')
axes[1].set_xticklabels([''])

# Violon
sns.violinplot(y=df['age_clean'].dropna(), ax=axes[2], color='steelblue', inner='quartile')
axes[2].set_title('Distribution de l\'âge — violon')
axes[2].set_ylabel('Âge')

plt.tight_layout()
plt.show()
print(df['age_clean'].describe().round(1))

**Lecture.** La distribution de l'âge des cyclistes accidentés est multimodale : un pic chez les jeunes (15-25 ans, étudiants, déplacements urbains) et un autre vers 30-50 ans (déplacements pendulaires). On note aussi une présence non négligeable de cyclistes seniors (60+), souvent vulnérables physiquement. La forme du violon révèle mieux que le boxplot la présence de plusieurs modes — le boxplot seul aurait pu suggérer une distribution simple unimodale.

#### Nombre de voies

In [ ]:
print(df['nbv_clean'].describe().round(2))
print()
print("Top valeurs :")
print(df['nbv_clean'].value_counts().sort_index().head(10))

**Lecture.** La majorité des accidents de vélo se produisent sur des routes à 2 voies, ce qui correspond à la voirie urbaine standard. Les routes à 4 voies et plus représentent un volume bien moindre, cohérent avec le fait que les cyclistes y sont rares.

### 7.2 Variables catégorielles

In [ ]:
def plot_categorical(var, mapping=None, top_n=None, figsize=(10, 4)):
    """Countplot + table de fréquences pour une variable catégorielle."""
    counts = df[var].value_counts(dropna=False)
    if top_n:
        counts = counts.head(top_n)

    fig, ax = plt.subplots(figsize=figsize)
    labels = [str(mapping.get(k, k)) if mapping else str(k) for k in counts.index]
    ax.bar(range(len(counts)), counts.values, color='steelblue')
    ax.set_xticks(range(len(counts)))
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_title(f'Distribution de `{var}`')
    ax.set_ylabel('Nb d\'usagers')
    plt.tight_layout()
    plt.show()

    freq = pd.DataFrame({'n': counts, 'pct (%)': (counts / counts.sum() * 100).round(2)})
    if mapping:
        freq.index = [mapping.get(k, k) for k in freq.index]
    return freq

display(plot_categorical('lum', mappings['lum']))

In [ ]:
plot_categorical('atm', mappings['atm'])

In [ ]:
plot_categorical('catr', mappings['catr'])

In [ ]:
plot_categorical('surf', mappings['surf'])

In [ ]:
plot_categorical('grav', mappings['grav'])

In [ ]:
plot_categorical('sexe', mappings['sexe'])

In [ ]:
plot_categorical('agg', mappings['agg'])

**Lectures principales.**
- **`lum`** : la grande majorité des accidents ont lieu en plein jour, ce qui reflète l'exposition (les cyclistes roulent davantage le jour), pas un risque relatif plus élevé.
- **`atm`** : les conditions "normales" dominent largement. La pluie n'apparaît pas comme un facteur dominant en volume — mais elle pourrait l'être en taux de gravité (à vérifier en bivarié).
- **`catr`** : les voies communales et routes départementales concentrent les accidents, conformément à la voirie cyclable.
- **`surf`** : surface "normale" très majoritaire ; les surfaces glissantes (mouillée, verglacée) restent rares mais à surveiller en bivarié.
- **`grav`** : forte dominance des "blessés légers", peu de tués (ce qui posera un problème de déséquilibre de classe pour la modélisation).
- **`sexe`** : majorité d'hommes — biais classique d'usage du vélo en France.
- **`agg`** : très large majorité d'accidents en agglomération.

### 7.3 Variables temporelles

In [ ]:
ordre_mois = ['janvier','février','mars','avril','mai','juin',
              'juillet','août','septembre','octobre','novembre','décembre']
ordre_jours = ['lundi','mardi','mercredi','jeudi','vendredi','samedi','dimanche']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Par année
df['an'].value_counts().sort_index().plot(kind='line', marker='o',
                                           ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Accidents par année')
axes[0,0].set_xlabel('Année')
axes[0,0].set_ylabel('Nb usagers')

# Par mois
df['mois'].value_counts().reindex(ordre_mois).plot(kind='bar',
                                                    ax=axes[0,1], color='steelblue')
axes[0,1].set_title('Accidents par mois')
axes[0,1].tick_params(axis='x', rotation=45)

# Par jour de semaine
df['jour'].value_counts().reindex(ordre_jours).plot(kind='bar',
                                                     ax=axes[1,0], color='steelblue')
axes[1,0].set_title('Accidents par jour de semaine')
axes[1,0].tick_params(axis='x', rotation=45)

# Par heure
df['heure'].value_counts().sort_index().plot(kind='bar',
                                              ax=axes[1,1], color='steelblue')
axes[1,1].set_title('Accidents par heure')
axes[1,1].set_xlabel('Heure')

plt.tight_layout()
plt.show()

**Lectures.**
- **Année** : tendance globalement croissante 2018-2024, en lien avec l'augmentation de la pratique du vélo (effet Covid notamment en 2020-2021).
- **Mois** : forte saisonnalité — pic d'accidents en juin-septembre, creux en hiver. La pratique du vélo elle-même est saisonnière, donc l'exposition explique en grande partie cette courbe.
- **Jour de semaine** : pic en milieu de semaine (lundi-vendredi), creux le dimanche. Cohérent avec les déplacements pendulaires domicile-travail.
- **Heure** : double pic 8h-9h et 17h-19h, classique des heures de pointe. Les accidents nocturnes sont rares en volume mais probablement plus graves (à vérifier en bivarié).

## 8. Analyse de la cible `grav`

### 8.1 Distribution des 4 classes

In [ ]:
grav_counts = df['grav'].value_counts().sort_index()
grav_pct = (grav_counts / len(df) * 100).round(2)

print("Distribution de grav :")
for k, v in grav_counts.items():
    label = mapping_grav.get(k, k)
    print(f"  {k} - {str(label):25s} : {v:>6} ({grav_pct[k]:5.2f}%)")

fig, ax = plt.subplots(figsize=(9, 4))
labels = [mapping_grav.get(k, k) for k in grav_counts.index]
colors = {'Indemne': '#2ecc71', 'Tué': '#c0392b',
          'Blessé hospitalisé': '#e67e22', 'Blessé léger': '#f1c40f'}
bar_colors = [colors.get(l, '#95a5a6') for l in labels]
bars = ax.bar(labels, grav_counts.values, color=bar_colors)
ax.set_title('Répartition de la gravité (2018-2024)')
ax.set_ylabel('Nombre d\'usagers')
for i, v in enumerate(grav_counts.values):
    ax.text(i, v, f'{grav_pct.iloc[i]}%', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

**Lecture.** La gravité est très déséquilibrée : les blessés légers représentent la grande majorité, les tués sont minoritaires (typiquement 1-3 % des cas). Les indemnes sont peu nombreux dans cette base — logique, car le BAAC n'enregistre que les accidents corporels avec au moins une victime, donc les indemnes sont les autres usagers du même accident.

### 8.2 Cible binaire `grave`

Selon l'énoncé du projet : **grave = (`grav` ∈ {2, 3})**, c'est-à-dire tué ou blessé hospitalisé.

In [ ]:
df['grave'] = df['grav'].isin([2, 3]).astype(int)

n_grave = df['grave'].sum()
n_total = len(df)
ratio = n_grave / n_total

print(f"Cas graves (tués + blessés hospit.) : {n_grave:,} ({ratio*100:.2f}%)")
print(f"Cas non graves                        : {n_total - n_grave:,} ({(1-ratio)*100:.2f}%)")
print(f"\nRatio de déséquilibre : 1 grave pour {(1-ratio)/ratio:.1f} non-graves")

**Conséquences pour la modélisation.**
- L'**accuracy** seule sera trompeuse : un modèle prédisant systématiquement "non grave" obtiendrait déjà ~80-85 % d'accuracy sans rien apprendre.
- Le **recall** sur la classe positive (graves) doit être la métrique prioritaire : l'objectif métier est de ne pas manquer les accidents graves.
- Outils de gestion du déséquilibre à tester : `class_weight='balanced'`, **SMOTE**, **undersampling**, seuil de décision ajusté sur la courbe précision-rappel.

### 8.3 Évolution de la cible dans le temps (vérification post-filtrage)

In [ ]:
grav_par_an = df.groupby('an')['grave'].agg(['sum', 'count', 'mean'])
grav_par_an.columns = ['n_graves', 'n_total', 'taux_grave']
grav_par_an['taux_grave'] = (grav_par_an['taux_grave'] * 100).round(2)
print(grav_par_an)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(grav_par_an.index, grav_par_an['taux_grave'], marker='o', color='#c0392b')
ax.axhline(df['grave'].mean()*100, ls='--', color='gray', alpha=0.5, label='Moyenne 2018-2024')
ax.set_title('Taux de gravité par année (après filtrage 2018+)')
ax.set_ylabel('% de cas graves')
ax.set_xlabel('Année')
ax.legend()
plt.tight_layout()
plt.show()

**Lecture.** Le taux de gravité est désormais relativement stable d'une année à l'autre, ce qui valide le choix du filtrage : les variations résiduelles sont faibles et compatibles avec une variabilité stochastique normale, et non plus avec une rupture méthodologique. On peut donc traiter les 7 années comme un échantillon homogène.

## 9. Analyse bivariée — variables × `grave`

### 9.1 Variables catégorielles vs `grave`

In [ ]:
def taux_gravite_par_modalite(var, mapping=None, min_n=30):
    """Tableau croisé : taux de gravité par modalité de la variable.
    On filtre les modalités avec moins de min_n observations (peu fiables)."""
    crosstab = pd.crosstab(df[var], df['grave'], margins=False)
    crosstab.columns = ['non_grave', 'grave']
    crosstab['total'] = crosstab.sum(axis=1)
    crosstab['pct_grave'] = (crosstab['grave'] / crosstab['total'] * 100).round(2)
    crosstab = crosstab[crosstab['total'] >= min_n].sort_values('pct_grave', ascending=False)
    if mapping:
        crosstab.index = [mapping.get(k, k) for k in crosstab.index]
    return crosstab

print("=== Luminosité ===")
display(taux_gravite_par_modalite('lum', mapping_lum))
print("\n=== Localisation (agg) ===")
display(taux_gravite_par_modalite('agg', mapping_agg))
print("\n=== Catégorie de route ===")
display(taux_gravite_par_modalite('catr', mapping_catr))
print("\n=== Conditions atmosphériques ===")
display(taux_gravite_par_modalite('atm', mapping_atm))

In [ ]:
def plot_taux_gravite(var, mapping=None, figsize=(10, 4), min_n=30):
    """Barplot du taux de gravité par modalité."""
    df_plot = taux_gravite_par_modalite(var, mapping, min_n=min_n)
    fig, ax = plt.subplots(figsize=figsize)
    ax.bar(df_plot.index.astype(str), df_plot['pct_grave'], color='#c0392b')
    ax.axhline(df['grave'].mean()*100, ls='--', color='gray',
               alpha=0.7, label=f'Taux global ({df["grave"].mean()*100:.1f}%)')
    ax.set_title(f'Taux de gravité par {var}')
    ax.set_ylabel('% de cas graves')
    ax.tick_params(axis='x', rotation=45)
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_taux_gravite('lum', mapping_lum)
plot_taux_gravite('atm', mapping_atm)
plot_taux_gravite('catr', mapping_catr, figsize=(11, 4))
plot_taux_gravite('agg', mapping_agg, figsize=(7, 4))
plot_taux_gravite('surf', mapping_surf, figsize=(11, 4))

**Lectures principales.**
- **Luminosité** : les accidents de nuit sans éclairage public sont nettement plus graves que ceux de jour. La visibilité réduite implique des chocs à plus haute énergie et des secours plus tardifs.
- **Atm** : pluie forte, brouillard, neige élèvent le taux de gravité par rapport au temps normal — quoique les volumes soient faibles (donc estimations bruitées).
- **Catégorie de route** : autoroute et route nationale sont **beaucoup plus graves** que voie communale (vitesses élevées, partage de la chaussée avec poids-lourds). Les voies communales urbaines, malgré leur volume élevé, sont les moins létales.
- **Agglo vs hors agglo** : les accidents hors agglo sont sensiblement plus graves (vitesses, isolement médical).
- **Surface** : verglacée, neige et corps gras augmentent la gravité.

> **Attention statistique.** Un *taux de gravité* est une proportion : pour les modalités à faible effectif (< 30 cas), l'estimation est bruitée. La fonction `taux_gravite_par_modalite` filtre déjà ces cas avec le paramètre `min_n=30`.

### 9.2 Variables numériques vs `grave`

#### Âge — boxplot + violon (le violon montre mieux la forme de la distribution)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot
df_plot = df[['age_clean', 'grave']].dropna()
df_plot['grave_label'] = df_plot['grave'].map({0: 'Non grave', 1: 'Grave'})
sns.boxplot(data=df_plot, x='grave_label', y='age_clean', ax=axes[0],
            palette={'Non grave': '#2ecc71', 'Grave': '#c0392b'})
axes[0].set_title('Âge selon la gravité — boxplot')
axes[0].set_xlabel('')
axes[0].set_ylabel('Âge')

# Violin plot
sns.violinplot(data=df_plot, x='grave_label', y='age_clean', ax=axes[1],
               palette={'Non grave': '#2ecc71', 'Grave': '#c0392b'},
               inner='quartile')
axes[1].set_title('Âge selon la gravité — violon')
axes[1].set_xlabel('')
axes[1].set_ylabel('Âge')

plt.tight_layout()
plt.show()

**Lecture du violon.** Le violon donne ce que le boxplot cache : la **forme** de la distribution. On observe que :
- Le groupe "non grave" a une distribution avec un mode marqué autour de 25-35 ans (jeunes actifs accidentés sans gravité majeure).
- Le groupe "grave" a une distribution **plus étalée vers le haut**, avec une densité accrue chez les 60+ ans. Cela traduit la plus grande vulnérabilité physique des seniors : un même choc produit des conséquences plus graves.

C'est un argument fort en faveur de l'inclusion de l'âge comme variable explicative dans les modèles.

#### Taux de gravité par tranche d'âge

In [ ]:
taux_age = df.groupby('age_tranche', observed=True)['grave'].agg(['mean', 'count'])
taux_age.columns = ['taux_grave', 'n']
taux_age['taux_grave'] = (taux_age['taux_grave'] * 100).round(2)
display(taux_age)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(taux_age.index.astype(str), taux_age['taux_grave'], color='#c0392b')
ax.axhline(df['grave'].mean()*100, ls='--', color='gray',
           alpha=0.7, label='Taux global')
ax.set_title('Taux de gravité par tranche d\'âge')
ax.set_ylabel('% de cas graves')
ax.set_xlabel('Tranche d\'âge')
ax.legend()
plt.tight_layout()
plt.show()

**Lecture.** Le taux de gravité augmente fortement avec l'âge, en particulier après 65 ans — confirmation de l'effet de vulnérabilité physique. Les enfants (<15) ont aussi un taux relativement élevé, possiblement lié à des chocs en zone résidentielle avec des véhicules circulant à allure modérée mais avec des cyclistes peu protégés.

## 10. Tests statistiques inférentiels

L'énoncé exige explicitement des "statistiques inférentielles" pour confirmer les liens observés visuellement.

### 10.1 Chi² + Cramér's V (catégorielles vs `grave`)

In [ ]:
def chi2_test(var):
    """Test du chi² + Cramér's V (taille d'effet)."""
    tab = pd.crosstab(df[var], df['grave'])
    if tab.shape[0] < 2 or tab.shape[1] < 2:
        return None
    chi2, pval, dof, _ = chi2_contingency(tab)
    n = tab.sum().sum()
    cramers_v = np.sqrt(chi2 / (n * (min(tab.shape) - 1)))
    return {'variable': var,
            'chi2': round(chi2, 2),
            'p_value': pval,
            'cramers_v': round(cramers_v, 3),
            'n': int(n)}

vars_cat = ['lum', 'atm', 'agg', 'int', 'col', 'catr', 'circ',
            'surf', 'infra', 'situ', 'sexe', 'trajet', 'prof', 'plan',
            'choc', 'obsm']
results = [chi2_test(v) for v in vars_cat if v in df.columns]
resultats_chi2 = pd.DataFrame([r for r in results if r is not None])
resultats_chi2 = resultats_chi2.sort_values('cramers_v', ascending=False)

display(resultats_chi2)

**Lecture critique.**

Avec 60 000+ observations, **presque tout devient "significatif"** (p < 0.05) — c'est le piège classique des grands échantillons. Le **Cramér's V** mesure la **taille d'effet** (entre 0 et 1, indépendant de n) et c'est lui qui doit guider l'interprétation :

- **V < 0.10** : effet négligeable (la variable explique très peu la gravité)
- **0.10 ≤ V < 0.20** : effet faible mais réel
- **0.20 ≤ V < 0.30** : effet modéré, intéressant pour la modélisation
- **V ≥ 0.30** : effet fort

**Conclusion pratique.** Les variables avec le plus haut Cramér's V sont les meilleures candidates pour la modélisation. À l'inverse, une variable significative mais avec V < 0.05 apporte peu et risque de surcharger le modèle sans gain de performance.

### 10.2 Mann-Whitney U pour les variables numériques

On utilise Mann-Whitney plutôt que le t-test parce que la distribution de l'âge n'est pas normale (test de Shapiro confirmerait, mais avec 60 000 obs il rejette presque systématiquement la normalité même pour des distributions quasi-normales — on se fie à l'inspection visuelle).

In [ ]:
# Âge vs grave
age_grave = df.loc[df['grave']==1, 'age_clean'].dropna()
age_non_grave = df.loc[df['grave']==0, 'age_clean'].dropna()
stat, pval = mannwhitneyu(age_grave, age_non_grave, alternative='two-sided')
print(f"=== Mann-Whitney U : âge ~ grave ===")
print(f"U statistic : {stat:,.0f}")
print(f"p-value     : {pval:.2e}")
print(f"Médiane âge graves     : {age_grave.median():.1f}")
print(f"Médiane âge non-graves : {age_non_grave.median():.1f}")
print(f"Différence médiane     : {age_grave.median() - age_non_grave.median():.1f} ans")

In [ ]:
# nbv vs grave
nbv_grave = df.loc[df['grave']==1, 'nbv_clean'].dropna()
nbv_non_grave = df.loc[df['grave']==0, 'nbv_clean'].dropna()
stat, pval = mannwhitneyu(nbv_grave, nbv_non_grave, alternative='two-sided')
print(f"\n=== Mann-Whitney U : nbv ~ grave ===")
print(f"U statistic : {stat:,.0f}")
print(f"p-value     : {pval:.2e}")
print(f"Médiane nbv graves     : {nbv_grave.median():.1f}")
print(f"Médiane nbv non-graves : {nbv_non_grave.median():.1f}")

**Lecture.** Le test confirme une différence statistique significative : les cyclistes accidentés gravement sont en moyenne plus âgés. La différence de médianes (en années) est l'**ampleur effective** de l'écart, plus parlante que la p-value seule.

Pour `nbv`, si la différence médiane est faible (< 1 voie), même si p < 0.05, l'effet pratique est probablement négligeable — encore l'effet de la taille d'échantillon.

## 11. Analyse temporelle approfondie

### 11.1 Heatmap heure × jour de semaine

In [ ]:
heatmap_data = df.groupby(['jour', 'heure']).size().unstack(fill_value=0)
heatmap_data = heatmap_data.reindex(ordre_jours)

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(heatmap_data, cmap='YlOrRd', ax=ax,
            cbar_kws={'label': 'Nb usagers accidentés'})
ax.set_title('Nombre d\'accidents par heure et jour de la semaine')
ax.set_xlabel('Heure')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

**Lecture.** On voit nettement les deux pics pendulaires (8-9h et 17-19h) en semaine, qui s'estompent le week-end. Le samedi montre un profil étalé sur toute la journée (loisirs), le dimanche est globalement plus calme.

### 11.2 Heatmap heure × jour pour le **taux de gravité** (et non le volume)

C'est plus utile pour identifier les créneaux à risque que pour identifier les créneaux de forte exposition.

In [ ]:
heatmap_grav = df.groupby(['jour', 'heure'])['grave'].mean().unstack() * 100
heatmap_grav = heatmap_grav.reindex(ordre_jours)

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(heatmap_grav, cmap='Reds', ax=ax,
            cbar_kws={'label': '% de cas graves'},
            vmin=0, vmax=heatmap_grav.max().max())
ax.set_title('Taux de gravité par heure et jour de la semaine')
ax.set_xlabel('Heure')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

**Lecture.** Les créneaux les plus dangereux ne sont pas ceux où il y a le plus d'accidents. Les heures nocturnes (0h-5h) et les week-ends en soirée concentrent les **taux** de gravité les plus élevés, alors qu'ils ont des volumes faibles. Cela cohère avec : visibilité réduite, vitesses plus élevées (routes vides), comportements à risque.

### 11.3 Heure × luminosité

In [ ]:
ct = pd.crosstab(df['heure'], df['lum'])
ct.columns = [mapping_lum.get(c, c) for c in ct.columns]
ct.plot(kind='bar', stacked=True, figsize=(14, 4), colormap='viridis')
plt.title('Conditions de luminosité par heure')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
plt.ylabel('Nb usagers')
plt.tight_layout()
plt.show()

**Lecture.** La luminosité est très corrélée à l'heure (sans surprise) — ce sera à traiter comme une **redondance** dans la matrice de corrélation. On évitera d'inclure les deux variables simultanément dans une régression logistique pour éviter la colinéarité.

### 11.4 Mois × atmosphère

In [ ]:
ct_atm = pd.crosstab(df['mois'], df['atm'], normalize='index') * 100
ct_atm.columns = [mapping_atm.get(c, c) for c in ct_atm.columns]
ct_atm = ct_atm.reindex(ordre_mois)
ct_atm.plot(kind='bar', stacked=True, figsize=(12, 4), colormap='Set3')
plt.title('Répartition (%) des conditions atmosphériques par mois')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
plt.ylabel('%')
plt.tight_layout()
plt.show()

**Lecture.** Saisonnalité attendue : la pluie est sur-représentée en automne/hiver, la neige uniquement en hiver. Néanmoins le temps "normal" reste majoritaire toute l'année dans ce dataset, signe que les cyclistes roulent peu par mauvais temps (effet d'auto-sélection).

## 12. Analyse géographique

### 12.1 Top départements

**Note méthodologique.** On dédoublonne sur `Num_Acc` ici : on compte des accidents, pas des usagers (sinon Paris serait artificiellement gonflé par les accidents impliquant plusieurs personnes).

In [ ]:
df_acc = df.drop_duplicates('Num_Acc')
top_dep = df_acc['dep'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 4))
top_dep.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Top 15 départements — nombre d\'accidents de vélo (2018-2024)')
ax.set_ylabel('Nb accidents')
ax.set_xlabel('Département')
plt.tight_layout()
plt.show()
print(top_dep)

**Lecture.** Les départements urbains denses (75-Paris, 13-Bouches-du-Rhône, 69-Rhône, 33-Gironde) dominent en volume, ce qui reflète l'**exposition** : ce sont les zones où l'on roule le plus à vélo. Cela ne signifie pas qu'il y est plus dangereux d'y rouler — pour cela, il faudrait normaliser par le nombre de cyclistes-kilomètres parcourus, donnée non disponible ici. **Biais à signaler dans la conclusion.**

### 12.2 Comparaison agglomération vs hors agglomération

In [ ]:
comp = df.groupby('agg').agg(
    n_accidents=('Num_Acc', 'nunique'),
    n_usagers=('grav', 'count'),
    taux_grave=('grave', 'mean')
)
comp['taux_grave'] = (comp['taux_grave'] * 100).round(2)
comp.index = [mapping_agg.get(i, i) for i in comp.index]
display(comp)

**Lecture.** Les accidents hors agglomération sont **beaucoup plus rares en volume** mais **nettement plus graves en taux**. Deux explications combinées :
1. Vitesses plus élevées hors agglomération
2. Délais d'intervention des secours plus longs

Cela fera de `agg` une variable importante du modèle.

### 12.3 Carte heatmap (folium)

In [ ]:
import folium
from folium.plugins import HeatMap

# On filtre les coordonnées valides en France métropolitaine
df_geo = df_acc[(df_acc['lat'].between(41, 51)) &
                (df_acc['long'].between(-5, 10))]
print(f"Accidents géolocalisés en France métropolitaine : {len(df_geo):,}")

# Heatmap centrée sur la France
m = folium.Map(location=[46.6, 2.5], zoom_start=6, tiles='cartodbpositron')
sample = df_geo[['lat', 'long']].sample(min(20000, len(df_geo)),
                                        random_state=RANDOM_STATE)
HeatMap(sample.values, radius=8, blur=10).add_to(m)
m

**Lecture.** La carte confirme la concentration des accidents en zones urbaines (Île-de-France, métropoles), ainsi que le long des grands axes. Elle révèle aussi des "zones blanches" (Massif central, Pyrénées) qui peuvent être de réelles zones à faible pratique cycliste — ou des zones de sous-déclaration à investiguer.

## 13. Corrélations entre variables explicatives

### 13.1 Pearson (relations linéaires) sur les variables numériques

In [ ]:
vars_num = ['age_clean', 'nbv_clean', 'an', 'heure', 'weekend', 'grave']
vars_num = [v for v in vars_num if v in df.columns]
corr_pearson = df[vars_num].corr(method='pearson')

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr_pearson, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=ax, vmin=-1, vmax=1, square=True)
ax.set_title('Corrélations Pearson (variables numériques)')
plt.tight_layout()
plt.show()

**Pearson** mesure les relations **linéaires**. C'est l'option par défaut mais elle est sensible aux distributions non normales et n'attrape pas les relations monotones non linéaires (ex. âge vs gravité, qui est plutôt en U que linéaire).

### 13.2 Spearman (relations monotones) — recommandé ici

In [ ]:
corr_spearman = df[vars_num].corr(method='spearman')

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr_spearman, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=ax, vmin=-1, vmax=1, square=True)
ax.set_title('Corrélations Spearman (variables numériques)')
plt.tight_layout()
plt.show()

**Pourquoi Spearman ici ?** Spearman se base sur les **rangs** plutôt que sur les valeurs. Il :
- est robuste aux distributions non normales (notre cas pour `age`),
- détecte toute relation **monotone** même non linéaire,
- est moins sensible aux valeurs extrêmes.

**Comparaison Pearson vs Spearman :** si les deux donnent des résultats proches, c'est que les relations sont à peu près linéaires. Si Spearman est sensiblement plus élevé que Pearson en valeur absolue, c'est le signe d'une relation **monotone non linéaire** (ex. courbe en S, exponentielle), et Spearman est alors plus fiable.

### 13.3 Cramér's V entre variables catégorielles

Pour repérer les **redondances** (ex. luminosité ↔ heure, agg ↔ catr) avant la modélisation.

In [ ]:
def cramers_v(x, y):
    """Cramér's V : équivalent d'une corrélation pour 2 variables catégorielles."""
    tab = pd.crosstab(x, y)
    if tab.shape[0] < 2 or tab.shape[1] < 2:
        return np.nan
    chi2 = chi2_contingency(tab)[0]
    n = tab.sum().sum()
    return np.sqrt(chi2 / (n * (min(tab.shape) - 1)))

vars_cat_corr = ['lum', 'atm', 'agg', 'int', 'col', 'catr', 'circ',
                 'surf', 'infra', 'situ', 'sexe', 'trajet']
vars_cat_corr = [v for v in vars_cat_corr if v in df.columns]

matrice_v = pd.DataFrame(index=vars_cat_corr, columns=vars_cat_corr, dtype=float)
for v1 in vars_cat_corr:
    for v2 in vars_cat_corr:
        if v1 == v2:
            matrice_v.loc[v1, v2] = 1.0
        else:
            sub = df[[v1, v2]].dropna()
            matrice_v.loc[v1, v2] = cramers_v(sub[v1], sub[v2])

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(matrice_v.astype(float), annot=True, fmt='.2f', cmap='YlOrRd',
            ax=ax, vmin=0, vmax=1, square=True)
ax.set_title("Cramér's V entre variables catégorielles")
plt.tight_layout()
plt.show()

**Lecture.** Toute paire avec V > 0.4 indique une redondance importante à gérer en modélisation :
- `agg` ↔ `catr` : très liées (en agglo on est sur voie communale, hors agglo sur départementale/nationale).
- `int` ↔ `infra` : intersections aménagées vs infrastructures de carrefour.
- `lum` aura un V élevé avec `heure` (calculable séparément) — confirmation visuelle déjà observée plus haut.

**Implication pour la modélisation.** Pour la régression logistique, qui souffre de la colinéarité, on devra choisir l'une des deux variables redondantes ou les combiner (ex. interaction `agg*catr`). Pour les modèles à arbres (Random Forest, XGBoost), la redondance est moins problématique mais elle dilue l'importance des features.

### 13.4 Cramér's V de chaque variable catégorielle avec la cible `grave`

C'est ce graphique qui guide directement la sélection de variables pour la modélisation : plus le V est élevé, plus la variable a de pouvoir discriminant sur la gravité.

In [ ]:
v_with_target = []
for v in vars_cat_corr:
    sub = df[[v, 'grave']].dropna()
    v_with_target.append({
        'variable': v,
        'cramers_v_avec_grave': round(cramers_v(sub[v], sub['grave']), 3)
    })
df_v_target = pd.DataFrame(v_with_target).sort_values('cramers_v_avec_grave', ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(df_v_target['variable'], df_v_target['cramers_v_avec_grave'], color='steelblue')
ax.set_xlabel("Cramér's V avec `grave`")
ax.set_title("Pouvoir discriminant des variables catégorielles sur la gravité")
ax.axvline(0.10, ls='--', color='gray', alpha=0.5, label='Seuil "effet faible"')
ax.axvline(0.20, ls='--', color='orange', alpha=0.5, label='Seuil "effet modéré"')
ax.legend()
plt.tight_layout()
plt.show()
display(df_v_target.set_index('variable').sort_values('cramers_v_avec_grave', ascending=False))

**Lecture.** Les variables avec un V > 0.10 sur la cible sont les plus prometteuses pour la modélisation. Les variables avec V < 0.05 apportent peu et peuvent être écartées dès le feature engineering pour simplifier le modèle, sauf intérêt métier spécifique.

## 14. Synthèse, biais et limites

### 14.1 Principaux facteurs de risque identifiés

L'analyse exploratoire fait ressortir plusieurs facteurs cohérents avec la littérature en sécurité routière :

1. **Âge du cycliste** — courbe en J : risque accru pour les <15 ans et fortement croissant après 65 ans (vulnérabilité physique).
2. **Catégorie de route** — autoroute, route nationale et hors agglomération sont les plus létaux (vitesses élevées).
3. **Luminosité** — nuit sans éclairage public : taux de gravité multiplié.
4. **Conditions atmosphériques** — pluie forte, brouillard, neige aggravent (volumes faibles → estimations à confirmer).
5. **État de surface** — verglas, neige, corps gras.
6. **Type de collision** — à explorer plus finement, lié au point de choc et à la manœuvre.

### 14.2 Biais et limites du jeu de données

- **Sous-déclaration.** Le BAAC n'enregistre que les accidents corporels avec intervention des forces de l'ordre. Les chutes seules sans intervention médicale formelle ou les accidents bénins non déclarés sont absents. La proportion de cas graves est donc artificiellement élevée — un cycliste qui tombe seul et se relève n'apparaît jamais dans ce fichier.
- **Biais géographique.** Les zones rurales sont sous-représentées : moins d'accidents reportés (moins d'agents sur place, plus de petits accidents non déclarés). La localisation manque pour ~10 % des observations.
- **Pas de dénominateur d'exposition.** On ne sait pas combien de cyclistes-kilomètres ont été parcourus dans chaque département, à chaque heure, par chaque tranche d'âge. **Le nombre brut d'accidents traduit donc autant l'exposition que le risque.** Pour une analyse de risque pure, il faudrait croiser avec des données de pratique cycliste (Vélo & Territoires, comptages métropolitains).
- **Évolutions BAAC dans le temps.** Filtrer à partir de 2018 résout la rupture majeure 2018-2019, mais des évolutions mineures peuvent subsister (ex. ajout des EDP en 2019). À documenter dans la modélisation.
- **Données usager-véhicule.** Une ligne ≠ un accident. La distinction doit être respectée à chaque agrégation : `Num_Acc` pour des comptages d'accidents, lignes brutes pour des analyses individuelles (gravité par usager, équipement).
- **Variables conditionnelles.** `obs`, `obsm`, `choc`, `manv` sont souvent vides quand sans objet — ne pas les imputer naïvement avec une valeur médiane.
- **Cible déséquilibrée.** ~15-20 % de cas graves : nécessitera des techniques spécifiques en modélisation (recall comme métrique principale, class_weight, SMOTE).

### 14.3 Implications pour la modélisation

- **Variables à inclure prioritairement** (Cramér's V > 0.10 ou Mann-Whitney clair) : `agg`, `catr`, `lum`, `surf`, `atm`, `age_tranche`, `obsm`, `int`.
- **Variables à arbitrer pour cause de redondance** : `lum` vs `heure`, `agg` vs `catr`, `int` vs `infra`.
- **Encodage catégoriel** : One-Hot pour les variables faiblement modales (`agg`, `sexe`), regroupement de modalités rares (< 1 % des observations) avant encodage pour éviter l'explosion dimensionnelle.
- **Métriques de validation** : recall sur les cas graves comme métrique principale, F1-score, ROC-AUC, et **precision-recall AUC** (plus pertinente que ROC-AUC en classes très déséquilibrées).
- **Validation croisée stratifiée** pour préserver la proportion de cas graves dans les folds.

---

*Fin de l'analyse exploratoire. Les sections suivantes du projet — feature engineering, modélisation (régression logistique, Random Forest, XGBoost), évaluation et interprétation — exploiteront le DataFrame `df` nettoyé et les conclusions ci-dessus pour construire un modèle prédictif de la gravité.*